## Imports

In [1]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../'))

from neuro_fuzzy_toolbox import ANFIS, Hybrid_learning_algorithm, alt_SONFIS, EarlyStopping, get_measures, Gaussian_MF

In [2]:
import numpy as np

In [3]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [4]:
from ucimlrepo import fetch_ucirepo

# Heart Disease

In [5]:
heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

In [6]:
X = X.fillna(value=0)

In [7]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    float64
 12  thal      303 non-null    float64
dtypes: float64(3), int64(10)
memory usage: 30.9 KB


In [8]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

In [9]:
unique_classes, counts = np.unique(y_train.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1 2 3 4]
[115  38  25  25   9]


In [10]:
unique_classes, counts = np.unique(y_test.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1 2 3 4]
[49 17 11 10  4]


In [11]:
scaler = MinMaxScaler(feature_range=(-1, 1))

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [12]:
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

In [13]:
loader = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size = 16, shuffle = True)
x_train = loader.dataset.tensors[0]
y_train = loader.dataset.tensors[1]
x_train.shape, y_train.shape

(torch.Size([212, 13]), torch.Size([212]))

In [14]:
x_train

tensor([[ 0.0000, -1.0000,  1.0000,  ...,  0.0000, -1.0000, -0.1429],
        [-0.3750,  1.0000, -0.3333,  ..., -1.0000, -1.0000, -0.1429],
        [ 0.2917,  1.0000,  1.0000,  ...,  0.0000,  0.3333,  1.0000],
        ...,
        [-0.3750,  1.0000, -0.3333,  ..., -1.0000, -1.0000, -0.1429],
        [ 0.8750, -1.0000, -0.3333,  ..., -1.0000, -0.3333, -0.1429],
        [ 0.1667,  1.0000,  1.0000,  ...,  0.0000, -0.3333,  1.0000]],
       dtype=torch.float64)

In [15]:
y_train

tensor([0, 0, 4, 0, 1, 2, 3, 0, 0, 0, 0, 0, 1, 2, 1, 2, 1, 0, 0, 3, 1, 0, 2, 0,
        0, 2, 2, 2, 0, 0, 0, 2, 3, 3, 0, 0, 0, 3, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0,
        0, 4, 0, 1, 0, 1, 0, 0, 0, 0, 0, 2, 0, 1, 0, 1, 0, 1, 0, 3, 1, 0, 0, 0,
        0, 0, 1, 0, 3, 0, 3, 1, 0, 0, 2, 3, 2, 0, 3, 1, 2, 2, 1, 0, 0, 0, 1, 1,
        4, 0, 0, 0, 4, 3, 3, 1, 0, 0, 1, 0, 2, 0, 0, 0, 3, 0, 0, 2, 0, 2, 0, 2,
        1, 3, 2, 0, 1, 1, 0, 2, 0, 0, 3, 0, 0, 3, 3, 0, 0, 0, 3, 1, 1, 0, 1, 2,
        0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 2, 2, 0, 0, 3, 0, 0, 3, 0, 0, 3, 0, 3, 0,
        1, 0, 0, 0, 0, 1, 0, 0, 3, 0, 0, 0, 4, 0, 0, 0, 2, 0, 1, 1, 4, 2, 1, 0,
        0, 4, 1, 0, 1, 0, 2, 0, 0, 3, 0, 0, 4, 0, 1, 0, 4, 0, 0, 3])

## Model & Training

### ANFIS

In [16]:
model = ANFIS(
    input_size = 13,
    fuzzy_rules = 1,
    outputs = 5,
    rule_reduced = True,
    output_type='multiclass'
)

model.init_premises(x_train)

### Hybrid Learning Algorithm

In [17]:
loss_fn = nn.functional.cross_entropy
epochs = 50
optimizer = torch.optim.AdamW
params = {'lr': 0.001, 'weight_decay': 0.01}
#optimizer = torch.optim.Adam
#params = {'lr': 0.005}
#optimizer = torch.optim.SGD
#params = {'lr': 0.001, 'momentum': 0.9}

early_stopping = EarlyStopping(
    patience=10, 
    delta=0.01
)

In [18]:
trainer = Hybrid_learning_algorithm(
    epochs=epochs,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    early_stopping=early_stopping
)

### SONFIS

In [19]:
Ngrow = 30
dGrow = 0.8
Nsplit = 30
eSplit = 0.7
Nvanish = 8
lVanish = 3

max_iterations = 50

anfis_trainer = trainer

validation = 0.2
sonfis_early_stopping = EarlyStopping(patience=6, delta=0.01)
last_training_iteration = True

In [20]:
sonfis = alt_SONFIS(
    Ngrow=Ngrow,
    dGrow=dGrow,
    Nsplit=Nsplit,
    eSplit=eSplit,
    Nvanish=Nvanish,
    lVanish=lVanish,
    max_iterations=max_iterations,
    ANFIStrainer=anfis_trainer,
    validation=validation,
    early_stopping=sonfis_early_stopping,
    last_training_iteration=last_training_iteration
)

In [21]:
%time sonfis(model, loader, verbose=True)

Iteration:  0/50 - loss: 2.619459 - validation loss: 2.519529
 -> Fuzzy rules: 2

Iteration:  1/50 - loss: 1.230890 - validation loss: 1.356032
 -> Fuzzy rules: 4

Iteration:  2/50 - loss: 1.155314 - validation loss: 1.403953
 -> Fuzzy rules: 6

Iteration:  3/50 - loss: 1.118116 - validation loss: 1.419237
 -> Fuzzy rules: 7

Iteration:  4/50 - loss: 1.162818 - validation loss: 1.409218
 -> Fuzzy rules: 8

Iteration:  5/50 - loss: 1.385843 - validation loss: 1.389899
 -> Fuzzy rules: 9

Iteration:  6/50 - loss: 3.198813 - validation loss: 1.326397
 -> Fuzzy rules: 10

Iteration:  7/50 - loss: 1.469967 - validation loss: 1.813941
 -> Fuzzy rules: 9

Iteration:  8/50 - loss: 8.125548 - validation loss: 1.845616
 -> Fuzzy rules: 10

Iteration:  9/50 - loss: 4.019040 - validation loss: 2.053383
 -> Fuzzy rules: 10

Iteration: 10/50 - loss: 3.463977 - validation loss: 1.855733
 -> Fuzzy rules: 10

Iteration: 11/50 - loss: 34.351406 - validation loss: 1.748267
 -> Fuzzy rules: 11

Early stop

In [22]:
test_measures = get_measures(model, x_test, y_test)

for measure in test_measures:
    print(measure + ':', test_measures[measure])

Accuracy: 0.4175824175824176
Precision: 0.49730850280300826
Recall: 0.4175824175824176
F1: 0.4442084078711986
Confusion Matrix: [[28 15  5  0  1]
 [ 6  5  4  2  0]
 [ 3  3  3  2  0]
 [ 0  3  5  2  0]
 [ 0  0  4  0  0]]


In [23]:
train_measures = get_measures(model, x_train, y_train)

for measure in train_measures:
    print(measure + ':', train_measures[measure])

Accuracy: 0.8160377358490566
Precision: 0.8199548572305129
Recall: 0.8160377358490566
F1: 0.8141238505347034
Confusion Matrix: [[104   5   5   1   0]
 [  5  29   1   3   0]
 [  0   1  22   1   1]
 [  2   3   5  14   1]
 [  1   3   0   1   4]]
